# Session 1: Data preparation

In [ ]:
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import time

from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split

from sklearn.model_selection import StratifiedKFold

le = LabelEncoder()

In [ ]:
url = "data/drebin.csv"

data = pd.read_csv(url)

data = data.sample(frac=1, random_state=42).reset_index(drop=True)
data = data.dropna()

data.head()

This dataset contains binary features indicating the presence or absence of specific Android permissions, API calls, and class usage for different applications. Each row represents an app, with the final column showing the class label (e.g. benign or malware). This structure is designed for malware classification tasks, helping to identify malicious apps based on their behavioral and permission patterns.

For more infomration on the dataset, please refer to this [paper](https://www.mlsec.org/docs/2014-ndss.pdf).

*Arp, Daniel, et al. "Drebin: Effective and explainable detection of android malware in your pocket." Ndss. Vol. 14. No. 1. 2014.*


The dataset can also be downloaded as csv here:

https://drive.google.com/file/d/1Cs2WOkpE1dFnGz4QAkKGcOwosoWWUAsb/view?usp=sharing

In [ ]:
# Show the shape of the dataset (rows, columns)
print("Shape of dataset:", data.shape)

# Check for missing values in each column
missing = data.isnull().sum()
print("\nMissing values per column:")
print(missing[missing > 0])


# Explore class distribution
print("\nClass distribution:")
print(data['class'].value_counts())

# Visualize class distribution
import matplotlib.pyplot as plt

data['class'].value_counts().plot(kind='bar')
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Class Distribution')
plt.grid(axis='y')
plt.show()


##  Parameters - Edit here


In [ ]:
# edit these features if you like

columns_of_interest = [
    'SEND_SMS', 'transact', 'WRITE_EXTERNAL_STORAGE'
]

## Run

In [ ]:
import matplotlib.pyplot as plt


# Plot one mini pie chart for each feature
fig, axes = plt.subplots(1, len(columns_of_interest), figsize=(4 * len(columns_of_interest), 4))

if len(columns_of_interest) == 1:
    axes = [axes]

for ax, col in zip(axes, columns_of_interest):
    counts = data[col].value_counts().sort_index()
    sizes = [counts.get(0, 0), counts.get(1, 0)]
    labels = ['0', '1']
    ax.pie(
        sizes,
        labels=labels,
        autopct='%1.1f%%',
        startangle=90,
        counterclock=False,
        wedgeprops={'edgecolor': 'white'}
    )
    ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
data.fillna(value=np.nan, inplace=True)

data = data[(data.astype(str) != '?').all(axis=1)]
print(len(data))

data = data.reset_index(drop = True)
print(len(data))

In [ ]:
feature_cols = [col for col in data.columns if col != 'class']
class_binary = (data['class'] != 'B').astype(int)
correlations = data[feature_cols].corrwith(class_binary)
correlations_sorted = correlations.abs().sort_values(ascending=False)

print("Top features correlated with being malicious:")
print(correlations_sorted.head(10))


In [ ]:
y = data.iloc[:,-1:]

y.head()
print(len(y))

In [ ]:
y = le.fit_transform(y)
print(len(y))

In [ ]:
X = data.iloc[:,:-1]
X.fillna(value=np.nan, inplace=True)
# print(X.isna().sum())
# print(X.isna().any().any())
# X = X.dropna()
X = np.array(X)
# print(len(X))


# Session 1: Logistic Regression


Logistic Regression is a simple and widely used classification algorithm that models the probability of a binary outcome based on input features. It uses the logistic (sigmoid) function to map predictions to probabilities between 0 and 1. Despite its simplicity, logistic regression often performs well as a baseline model for binary classification tasks and provides interpretable results.

In [ ]:
from sklearn.linear_model import LogisticRegression

from sklearn import metrics

from sklearn.model_selection import train_test_split

## Parameters - Edit here


In [ ]:
# the threshold to convert probabilities into binary predictions
pred_prob_threshold = 0.5

## Run




In [ ]:
def mal_detection_LR(Xtrain, Xtest, ytrain, ytest):
    LR_classifier = LogisticRegression()
    LR_classifier.fit(Xtrain, ytrain)  # Train the model

    pred_prob = LR_classifier.predict_proba(Xtest)[:, 1]  # Predict probabilities for the test data
    pred = (pred_prob > pred_prob_threshold)  # Apply threshold to convert probabilities into binary predictions

    fbeta = metrics.fbeta_score(ytest, pred, average='weighted', beta=10)  # Calculate F-beta score
    tn, fp, fn, tp = metrics.confusion_matrix(ytest, pred).ravel()  # Calculate confusion matrix

    tpr = tp / (tp + fn)  # Calculate true positive rate (TPR)
    fpr = fp / (fp + tn)  # Calculate false positive rate (FPR)

    return fbeta, tpr, fpr, ytest, pred_prob

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

start_time = time.time()

# Run the model and get metrics
fbeta, tpr, fpr, y_true, y_scores = mal_detection_LR(X_train, X_test, y_train, y_test)

print("--- %d minutes %d seconds ---" % (int((time.time() - start_time) // 60), int((time.time() - start_time) % 60)))

cvscores_LR = [(fbeta, tpr, fpr)]

print("F-Beta Score", fbeta)

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr_vals, tpr_vals, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr_vals, tpr_vals)

# Plot ROC
plt.figure()
plt.plot(fpr_vals, tpr_vals, label=f'ROC Curve (AUC = {roc_auc:.5f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

The Receiver Operating Characteristic (ROC) curve visualizes the performance of a binary classifier across all possible classification thresholds. It shows the trade-off between the true positive rate (sensitivity) and the false positive rate (1 - specificity). The closer the ROC curve is to the top-left corner, the better the model’s ability to distinguish between classes. The Area Under the Curve (AUC) quantifies overall performance — an AUC of 1 indicates perfect classification, while an AUC of 0.5 suggests random guessing.

In [ ]:
pd.DataFrame(cvscores_LR, columns= ['F-beta Score', 'TPR', "FPR"])


# Session 2: Support Vector Machine

Support Vector Machine (SVM) is a powerful supervised learning algorithm for classification tasks. SVM finds the best boundary (hyperplane) that separates different classes in the feature space. It is effective in high-dimensional spaces and can model non-linear relationships using different kernel functions, such as the radial basis function (rbf).

In [ ]:
from sklearn.svm import SVC

split_random_state = 42

## Parameters - Edit here

In [ ]:
# SVM params
svm_kernel = 'sigmoid'      # Possible: 'linear', 'poly', 'rbf', 'sigmoid'
svm_C = 1                 # Typical: 0.01, 0.1, 1.0, 10, 100 (positive float, regularization strength)
svm_gamma = 'auto'        # Possible: 'scale', 'auto', or any positive float (e.g., 0.01, 0.1, 1)

# Classification threshold
pred_threshold = 0.5       # Typical: 0.1 to 0.9 (float between 0 and 1, for binary decision boundary)

# Data split params
test_size = 0.2            # Typical: 0.2 (20% test), can be any float between 0 and 1


## Run

In [ ]:
def mal_detection_SVM(Xtrain, Xtest, ytrain, ytest,
                      kernel=svm_kernel, C=svm_C, gamma=svm_gamma, probability=True, threshold=pred_threshold):

    SVM_classifier = SVC(kernel=kernel, C=C, gamma=gamma, probability=probability)
    SVM_classifier.fit(Xtrain, ytrain)

    pred_prob = SVM_classifier.predict_proba(Xtest)[:, 1]
    pred = (pred_prob > threshold)

    fbeta = metrics.fbeta_score(ytest, pred, average='weighted', beta=10)
    tn, fp, fn, tp = metrics.confusion_matrix(ytest, pred).ravel()
    tpr = tp / (tp + fn)
    fpr = fp / (fp + tn)

    return fbeta, tpr, fpr, ytest, pred_prob

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, stratify=y, random_state=split_random_state
)

start_time = time.time()

# Run SVM
fbeta, tpr, fpr, y_true, y_scores = mal_detection_SVM(X_train, X_test, y_train, y_test)

print(f"F-beta score: {fbeta:.3f}")
print(f"True Positive Rate (TPR): {tpr:.3f}")
print(f"False Positive Rate (FPR): {fpr:.3f}")
print("--- %d minutes %d seconds ---" % (int((time.time() - start_time) // 60), int((time.time() - start_time) % 60)))

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr_vals, tpr_vals, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr_vals, tpr_vals)

# Plot ROC
plt.figure()
plt.plot(fpr_vals, tpr_vals, label=f'ROC Curve (AUC = {roc_auc:.5f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.decomposition import PCA

# Reduce to 2D for visualization using PCA
pca = PCA(n_components=2)
X_vis = pca.fit_transform(X_train)
X_test_vis = pca.transform(X_test)

# Train a new SVM on the 2D data for plotting
svm_vis = SVC(kernel=svm_kernel, C=svm_C, gamma=svm_gamma, probability=True)
svm_vis.fit(X_vis, y_train)

# Create mesh grid
x_min, x_max = X_vis[:, 0].min() - 1, X_vis[:, 0].max() + 1
y_min, y_max = X_vis[:, 1].min() - 1, X_vis[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300)
)

# Predict over mesh
Z = svm_vis.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot decision boundary
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
plt.scatter(
    X_vis[:, 0], X_vis[:, 1],
    c=y_train, cmap=plt.cm.coolwarm, edgecolors='k', s=20, alpha=0.6
)
plt.title(f'SVM Decision Boundary (kernel={svm_kernel})')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.show()